# BraTS 2021 Preprocessing Pipeline — All Fixes Applied
**SEIS 766 Vision AI — Spring 2026**

Corrected task ordering: `SEG-01 → SEG-02 → SEG-03 → SEG-06 → SEG-04 → SEG-05`

All fixes from code review applied:
1. SEG-01: Spot-check 3 random cases with shape verification
2. SEG-03: `np.nan_to_num()` before normalization
3. SEG-06: `.copy()` before shuffle to protect caller's list
4. SEG-06: Check if split JSON already exists
5. SEG-04: `+1` on end_z to include full margin
6. SEG-04: Partition-aware extraction (train/val saved separately)
7. SEG-05: `.copy()` after `np.flip` for array contiguity
8. SEG-05: Geometric and photometric transforms separated
9. SEG-05: `borderMode`/`borderValue` in warpAffine
10. SEG-05: Mask cast to float32 before warp, uint8 after
11. SEG-05: Wrapped in Dataset class (train augments, val doesn't)
12. Execution: leakage verification check


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/"


In [ ]:
import os
import json
import random
import time
import numpy as np
import cv2
import nibabel as nib
from concurrent.futures import ProcessPoolExecutor, as_completed

# ═══════════════════════════════════════════════════════════════
# PATHS
#
# DRIVE_DATA: where raw BraTS folders live on Google Drive (read)
# LOCAL_DATA: local SSD copy of raw data (fast read)
# LOCAL_DIR:  local SSD for extracted slices (fast write)
# OUTPUT_DIR: Drive folder for persistent files (split JSON,
#             model checkpoints, zip of slices)
#
# STRATEGY: Copy raw data from Drive → local SSD ONCE at the
# start. All processing reads/writes locally. Only final
# artifacts (zip, models, JSON) go back to Drive.
# ═══════════════════════════════════════════════════════════════

DRIVE_DATA = ROOT + "Group_Project_766"         # raw data on Drive
LOCAL_DATA = "/content/brats_raw"                # local copy of raw data
LOCAL_DIR  = "/content/local_slices"             # extracted slices
OUTPUT_DIR = ROOT + "Group_Project_766"          # persistent Drive storage
TRAIN_PATH = LOCAL_DATA                          # processing reads from local

print(f"Drive data exists: {os.path.exists(DRIVE_DATA)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Copy Raw BraTS Data from Drive → Local SSD
#
# WHY: Google Drive FUSE mount is slow for random file access.
# Reading 1,251 cases × 5 NIfTI files = 6,255 network reads.
# Copying everything to local SSD first makes each nib.load()
# read from a physical disk instead of over the network.
#
# This takes 5-10 minutes but saves 60+ minutes during
# extraction because every subsequent read is 10-100x faster.
#
# SKIP THIS CELL if data was already copied in this session.
# ═══════════════════════════════════════════════════════════════

if os.path.exists(LOCAL_DATA) and len(os.listdir(LOCAL_DATA)) > 1000:
    count = len([d for d in os.listdir(LOCAL_DATA) if d.startswith('BraTS2021')])
    print(f"✅ Local data already exists: {count} cases in {LOCAL_DATA}")
else:
    print(f"📥 Copying raw BraTS data from Drive to local SSD...")
    print(f"   From: {DRIVE_DATA}")
    print(f"   To:   {LOCAL_DATA}")
    print(f"   This takes 5-10 minutes but saves 60+ minutes later.")

    start = time.time()
    os.makedirs(LOCAL_DATA, exist_ok=True)

    # Use shell cp for bulk copy — faster than Python shutil for
    # thousands of files over FUSE mount
    !cp -r "{DRIVE_DATA}"/BraTS2021_* "{LOCAL_DATA}/"

    elapsed = time.time() - start
    count = len([d for d in os.listdir(LOCAL_DATA) if d.startswith('BraTS2021')])
    print(f"\n✅ Copied {count} cases in {elapsed/60:.1f} minutes")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-01: Verify BraTS 2021 Dataset (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

def verify_data():
    if not os.path.exists(TRAIN_PATH):
        print(f"❌ Folder not found at: {TRAIN_PATH}")
        return []

    cases = sorted([d for d in os.listdir(TRAIN_PATH)
                    if d.startswith('BraTS2021') and
                    os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"📂 Found {len(cases)} training cases in {TRAIN_PATH}")

    if cases:
        spot_checks = random.sample(cases, min(3, len(cases)))
        suffixes = ['_t1.nii.gz','_t1ce.nii.gz','_t2.nii.gz','_flair.nii.gz','_seg.nii.gz']
        for cid in spot_checks:
            for s in suffixes:
                fp = os.path.join(TRAIN_PATH, cid, f"{cid}{s}")
                if not os.path.exists(fp):
                    print(f"  ⚠️ Missing: {fp}")
                else:
                    img = nib.load(fp)
                    ok = '✅' if img.shape == (240,240,155) else '⚠️'
                    print(f"  {ok} {cid}{s} → {img.shape}")
    return cases

all_cases = verify_data()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-02: NIfTI Loading Pipeline
# ═══════════════════════════════════════════════════════════════

def load_case_data(case_id, data_path=None):
    """Load all 4 modalities + seg mask for one case."""
    if data_path is None:
        data_path = TRAIN_PATH
    case_path = os.path.join(data_path, case_id)
    data_dict = {}
    for mod in ['t1', 't1ce', 't2', 'flair']:
        fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
        data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
    seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
    data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)
    return data_dict

if all_cases:
    s = load_case_data(all_cases[0])
    print(f"✅ Loaded {all_cases[0]}: " +
          ', '.join(f"{k}={v.shape}" for k,v in s.items()))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-03: Per-Volume Z-Score Normalization
# FIX: np.nan_to_num() before computing statistics
# ═══════════════════════════════════════════════════════════════

def normalize_volume(volume):
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)
    brain_mask = volume > 0
    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()
        normalized = (volume - mean) / (std + 1e-8)
        normalized[~brain_mask] = 0
        return normalized.astype(np.float32)
    return volume.astype(np.float32)

def normalize_case(data_dict):
    for mod in ['t1', 't1ce', 't2', 'flair']:
        data_dict[mod] = normalize_volume(data_dict[mod])
    return data_dict

if all_cases:
    norm = normalize_volume(s['flair'])
    t = norm[norm != 0]
    print(f"📊 Normalization: mean={t.mean():.4f}, std={t.std():.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-06: Case-Level Train/Val Split
# FIX: .copy() before shuffle
# FIX: Check if split JSON already exists
# Split JSON saves to Drive (persistent across sessions)
# ═══════════════════════════════════════════════════════════════

def save_dataset_split(case_ids, train_ratio=0.8):
    random.seed(42)
    shuffled = case_ids.copy()
    random.shuffle(shuffled)
    split_point = int(len(shuffled) * train_ratio)
    split_dict = {'train': shuffled[:split_point], 'val': shuffled[split_point:]}

    output_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
    try:
        with open(output_path, 'w') as f:
            json.dump(split_dict, f, indent=4)
        print(f"✅ Split saved to: {output_path}")
    except OSError as e:
        print(f"❌ Failed: {e}")
    print(f"   Training: {len(split_dict['train'])}, Validation: {len(split_dict['val'])}")
    return split_dict

split_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
if os.path.exists(split_path):
    print(f"📄 Loading existing split...")
    with open(split_path) as f:
        dataset_split = json.load(f)
    print(f"   {len(dataset_split['train'])} train / {len(dataset_split['val'])} val")
else:
    dataset_split = save_dataset_split(all_cases)

overlap = set(dataset_split['train']) & set(dataset_split['val'])
print(f"{'❌ LEAKAGE!' if overlap else '✅ No leakage'}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-04: Partition-Aware Slice Extraction — PARALLELIZED
#
# OPTIMIZATION: Uses concurrent.futures.ProcessPoolExecutor
# to process multiple cases simultaneously across CPU cores.
#
# On an A100 High-RAM instance (8-12 cores), this processes
# 4-8 cases at once instead of 1, cutting extraction time
# from ~90 minutes to ~15-20 minutes.
#
# All reads come from LOCAL_DATA (local SSD copy).
# All writes go to LOCAL_DIR (local SSD).
# ═══════════════════════════════════════════════════════════════

def extract_tumor_slices(data_dict, margin=5):
    seg = data_dict['seg']
    z_indices = np.any(seg > 0, axis=(0, 1))
    if not np.any(z_indices):
        return [], []
    start_z = max(0, np.where(z_indices)[0].min() - margin)
    end_z = min(seg.shape[2], np.where(z_indices)[0].max() + margin + 1)
    images, masks = [], []
    for z in range(start_z, end_z):
        combined = np.stack([
            data_dict['t1'][:,:,z], data_dict['t1ce'][:,:,z],
            data_dict['t2'][:,:,z], data_dict['flair'][:,:,z]
        ], axis=0)
        images.append(combined)
        masks.append(seg[:,:,z])
    return images, masks


def process_single_case(args):
    """Process one case: load → normalize → extract → save.
    This function runs in a separate process via ProcessPoolExecutor.
    Must be a top-level function (not a closure) for pickling."""
    case_id, data_path, save_dir = args

    try:
        # Load from local SSD
        case_path = os.path.join(data_path, case_id)
        data_dict = {}
        for mod in ['t1', 't1ce', 't2', 'flair']:
            fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
            data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
        seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
        data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)

        # Normalize per-volume
        for mod in ['t1', 't1ce', 't2', 'flair']:
            vol = np.nan_to_num(data_dict[mod], nan=0.0, posinf=0.0, neginf=0.0)
            brain = vol > 0
            if np.any(brain):
                m, s = vol[brain].mean(), vol[brain].std()
                vol = (vol - m) / (s + 1e-8)
                vol[~brain] = 0
            data_dict[mod] = vol.astype(np.float32)

        # Extract slices
        seg = data_dict['seg']
        z_idx = np.any(seg > 0, axis=(0, 1))
        if not np.any(z_idx):
            return case_id, 0

        start_z = max(0, np.where(z_idx)[0].min() - 5)
        end_z = min(seg.shape[2], np.where(z_idx)[0].max() + 6)

        count = 0
        for z in range(start_z, end_z):
            img = np.stack([
                data_dict['t1'][:,:,z], data_dict['t1ce'][:,:,z],
                data_dict['t2'][:,:,z], data_dict['flair'][:,:,z]
            ], axis=0)
            msk = seg[:,:,z]
            np.savez_compressed(
                os.path.join(save_dir, f"{case_id}_slice_{count:03d}.npz"),
                image=img, mask=msk)
            count += 1

        return case_id, count
    except Exception as e:
        return case_id, -1  # signal error


def extract_partition_parallel(case_ids, partition_name, num_workers=6):
    """Extract slices using multiple CPU cores in parallel."""
    save_dir = os.path.join(LOCAL_DIR, 'slices', partition_name)
    os.makedirs(save_dir, exist_ok=True)

    # Build argument list for each case
    args_list = [(cid, LOCAL_DATA, save_dir) for cid in case_ids]

    total_slices = 0
    completed = 0
    errors = []
    start_time = time.time()

    print(f"  [{partition_name}] Processing {len(case_ids)} cases "
          f"with {num_workers} parallel workers...")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_case, a): a[0]
                   for a in args_list}

        for future in as_completed(futures):
            case_id, count = future.result()
            completed += 1
            if count < 0:
                errors.append(case_id)
            else:
                total_slices += count

            if completed % 100 == 0 or completed == len(case_ids):
                elapsed = time.time() - start_time
                rate = completed / elapsed * 60
                print(f"  [{partition_name}] {completed}/{len(case_ids)} cases, "
                      f"{total_slices} slices, {elapsed/60:.1f}min "
                      f"({rate:.0f} cases/min)")

    elapsed = time.time() - start_time
    print(f"✅ {partition_name}: {total_slices} slices in {elapsed/60:.1f} minutes")
    if errors:
        print(f"⚠️  {len(errors)} cases had errors: {errors[:5]}...")

    return total_slices

print("✅ Extraction functions ready (parallelized)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run Parallel Extraction
#
# Estimated time: 10-20 minutes on A100 High-RAM (8+ cores)
# vs 90+ minutes single-threaded.
#
# Adjust NUM_WORKERS based on your CPU cores.
# A100 High-RAM typically has 12 cores → use 8 workers
# (leave some headroom for system processes).
#
# SKIP THIS CELL if slices already exist (run Cell 11 instead).
# ═══════════════════════════════════════════════════════════════

# Check available cores
import multiprocessing
num_cores = multiprocessing.cpu_count()
NUM_WORKERS = min(8, num_cores - 1)  # leave 1 core for system
print(f"💻 Available CPU cores: {num_cores}, using {NUM_WORKERS} workers")

total_start = time.time()

print("\n📤 Extracting training slices...")
train_count = extract_partition_parallel(
    dataset_split['train'], 'train', num_workers=NUM_WORKERS)

print("\n📤 Extracting validation slices...")
val_count = extract_partition_parallel(
    dataset_split['val'], 'val', num_workers=NUM_WORKERS)

total_elapsed = time.time() - total_start
print(f"\n✅ TOTAL: {train_count} train + {val_count} val slices "
      f"in {total_elapsed/60:.1f} minutes")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Transfer Slices to Drive (one large zip file)
# ═══════════════════════════════════════════════════════════════

import shutil

zip_path = '/content/brats_slices'
print('📦 Zipping extracted slices...')
shutil.make_archive(zip_path, 'zip', LOCAL_DIR)
print(f'✅ Created {zip_path}.zip')

drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
print(f'📤 Copying to Drive...')
shutil.copy2(f'{zip_path}.zip', drive_zip)
print(f'✅ Saved: {drive_zip} ({os.path.getsize(drive_zip)/1e9:.2f} GB)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Slices from Drive Zip (future sessions)
#
# Run THIS cell instead of Cells 3 + 9 + 10 if slices were
# already extracted in a previous session.
# ═══════════════════════════════════════════════════════════════

import zipfile

LOCAL_DIR = '/content/local_slices'
drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
local_check = os.path.join(LOCAL_DIR, 'slices', 'train')

if os.path.exists(local_check):
    nt = len([f for f in os.listdir(local_check) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Local slices exist: {nt} train + {nv} val')
elif os.path.exists(drive_zip):
    print(f'📥 Extracting from Drive zip...')
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(LOCAL_DIR)
    nt = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','train')) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Unzipped: {nt} train + {nv} val')
else:
    print('⚠️  No zip on Drive. Run extraction cells first.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-05: Train-Only Augmentation
#
# FIX: .copy() after np.flip
# FIX: Geometric (image+mask) separated from photometric (image only)
# FIX: borderMode=BORDER_CONSTANT, borderValue=0.0
# FIX: Mask cast float32 before warp, uint8 after
# FIX: Wrapped in Dataset class with augment flag
# ═══════════════════════════════════════════════════════════════

def geometric_augmentation(image, mask):
    """GEOMETRIC: apply to BOTH image and mask. NN interp on mask."""
    if random.random() > 0.5:
        image = np.flip(image, axis=2).copy()  # FIX: .copy()
        mask = np.flip(mask, axis=1).copy()     # FIX: .copy()

    angle = random.uniform(-15, 15)
    h, w = mask.shape
    rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    aug_img = np.zeros_like(image)
    for c in range(image.shape[0]):
        aug_img[c] = cv2.warpAffine(
            image[c], rot, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

    aug_msk = cv2.warpAffine(
        mask.astype(np.float32), rot, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0.0
    ).astype(np.uint8)

    return aug_img, aug_msk


def photometric_augmentation(image):
    """PHOTOMETRIC: IMAGE ONLY. Never apply to mask."""
    image = image * random.uniform(0.9, 1.1)
    if random.random() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = image + noise
    return image


def apply_augmentation(image, mask):
    """Geometric first (both), then photometric (image only)."""
    image, mask = geometric_augmentation(image, mask)
    image = photometric_augmentation(image)
    return image, mask


class BraTSSliceDataset:
    """Dataset wrapper. augment=True for train, False for val."""
    def __init__(self, slice_dir, augment=False):
        self.slice_dir = slice_dir
        self.augment = augment
        self.file_list = sorted(
            [f for f in os.listdir(slice_dir) if f.endswith('.npz')])
        print(f"📦 {len(self.file_list)} slices "
              f"(augment={'ON' if augment else 'OFF'})")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.slice_dir, self.file_list[idx]))
        image = data['image'].astype(np.float32)
        mask = data['mask'].astype(np.uint8)

        if self.augment:
            image, mask = apply_augmentation(image, mask)

        return image.astype(np.float32), mask.astype(np.int64)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Create Datasets and Verify (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

train_dir = os.path.join(LOCAL_DIR, 'slices', 'train')
val_dir = os.path.join(LOCAL_DIR, 'slices', 'val')

train_dataset = BraTSSliceDataset(train_dir, augment=True)
val_dataset = BraTSSliceDataset(val_dir, augment=False)

if len(train_dataset) > 0:
    img_t, msk_t = train_dataset[0]
    print(f'\n🔬 Train: {img_t.shape} {img_t.dtype}, mask labels: {np.unique(msk_t)}')
if len(val_dataset) > 0:
    img_v, msk_v = val_dataset[0]
    print(f'🔬 Val:   {img_v.shape} {img_v.dtype}, mask labels: {np.unique(msk_v)}')
print('\n🚀 Phase 1 complete.')
